# 真实迁移系统：实验集 70 样本基线 v1

2026-11-18。使用当前真实云库、真实 Worker 与 Qwen 模型，策略 SSM＋有界修复，最多 3 轮。仅运行 experiment 子集，validation/test 不运行。没有预设交互场景，人工检查另行记录。

配置、代码和输入见 protocol.json、implementation/ 和 inputs/。下面只读取本批输出，不启动模型或改变原始结果；执行中统计为阶段性进度，不作为最终通过率。

In [ ]:
from pathlib import Path
import json, csv
batch_name = 'real_system_experiment70_v1_20261118'
batch = Path.cwd() if Path('protocol.json').is_file() else Path.cwd() / 'experiment_runs' / batch_name
if not (batch / 'protocol.json').is_file():
    batch = Path.cwd() / batch_name
summary = json.loads((batch / 'summary.json').read_text(encoding='utf-8'))
summary

In [ ]:
with (batch / 'sample_results.csv').open(encoding='utf-8-sig', newline='') as stream:
    sample_results = list(csv.DictReader(stream))
failures = [row for row in sample_results if row['status'] in ('failed', 'blocked', 'review_required', 'cancelled')]
[(row['component'], row['status'], row['stop_reason'], row['error_codes']) for row in failures]

In [ ]:
failure_groups = json.loads((batch / 'failure_groups.json').read_text(encoding='utf-8'))
[(group['category'], group['code'], len(group['samples'])) for group in failure_groups]

## 集中修复安排

等本批 70 项全部结束后，结合各样本 report.json、artifacts/evaluations/、各轮 San、规则与模型调用收据定位原因。按源基准／支持范围、语法运行、静态语义、结构控件、视觉样式和模型预算分组；先查误报和通用规则闭环，再决定生成与修复 Prompt 的修改。

每项通用修复需要冻结失败产物的重放和真实差异证据；修复后创建 v2 比较，保留 v1 原始结果。此单元未记录任何尚未执行的修复成功。